# GRPO Training — Poker Decision Advisor

Fine-tunes Qwen3-8B with GRPO (Group Relative Policy Optimization) on the PokerBench dataset.

**Setup:** Kaggle → Settings → Accelerator → GPU P100 → Internet → On

**SFT adapter:** Upload your SFT adapter folder to a Kaggle dataset, or download from Google Drive/HuggingFace Hub.

## 1. Install dependencies

In [ ]:
!pip install -q transformers datasets peft bitsandbytes trl accelerate
# If you used Unsloth for SFT, uncomment the next line:
# !pip install -q unsloth

## 2. GPU check

In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU found — enable GPU in Kaggle settings"
gpu = torch.cuda.get_device_properties(0)
print(f"GPU : {gpu.name}")
print(f"VRAM: {gpu.total_mem / 1e9:.1f} GB" if hasattr(gpu, 'total_mem') else f"VRAM: {gpu.total_memory / 1e9:.1f} GB")

## 3. Download SFT adapter

Choose ONE of the options below depending on where your adapter is stored.

In [ ]:
# ── OPTION A: Download from Google Drive ──
# Make sure the folder sharing is set to "Anyone with the link"

!pip install -q gdown
import gdown

GDRIVE_FOLDER_URL = "https://drive.google.com/drive/folders/YOUR_FOLDER_ID_HERE"  # ← CHANGE THIS
SFT_ADAPTER_DIR = "./sft-adapter"

gdown.download_folder(GDRIVE_FOLDER_URL, output=SFT_ADAPTER_DIR, quiet=False)
print("Downloaded SFT adapter from Google Drive")

In [ ]:
# ── OPTION B: If adapter is uploaded as a Kaggle dataset ──
# Uncomment and adjust the path:

# import shutil
# SFT_ADAPTER_DIR = "./sft-adapter"
# shutil.copytree("/kaggle/input/your-dataset-name/sft-adapter", SFT_ADAPTER_DIR)
# print("Copied SFT adapter from Kaggle dataset")

In [ ]:
# ── OPTION C: If adapter is on HuggingFace Hub ──
# Uncomment and set:

# SFT_ADAPTER_DIR = "dominicvdb/poker-sft-adapter"  # ← your HF repo ID
# print(f"Will load adapter from HuggingFace Hub: {SFT_ADAPTER_DIR}")

In [ ]:
# Verify adapter files exist
import os
if os.path.isdir(SFT_ADAPTER_DIR):
    print("Adapter files:", os.listdir(SFT_ADAPTER_DIR))
else:
    print(f"Will download from HuggingFace Hub: {SFT_ADAPTER_DIR}")

## 4. Load dataset

In [ ]:
from datasets import load_dataset

dataset = load_dataset("RZ412/PokerBench", cache_dir="./data")
train_ds = dataset["train"]
test_ds = dataset["test"]
print(f"Train: {len(train_ds):,}  Test: {len(test_ds):,}")
print(f"Columns: {train_ds.column_names}")
print(f"\nExample output: {train_ds[0]['output']}")

## 5. Preprocessor module (inlined)

In [ ]:
SYSTEM_PROMPT = (
    "You are a poker decision engine. Given a game scenario, output only the "
    "optimal action (check, fold, call, bet X, or raise X). Do not explain."
)


def format_sft(row: dict) -> list:
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": row["instruction"]},
        {"role": "assistant", "content": row["output"]},
    ]


def format_grpo(row: dict) -> list:
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": row["instruction"]},
    ]


def apply_chat_template(messages, tokenizer, add_generation_prompt=False):
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=add_generation_prompt,
    )


print("Preprocessor ready")

## 6. Reward function (inlined — tiered scoring)

In [ ]:
import re

VALID_ACTIONS = ("check", "fold", "call", "bet", "raise")
_THINK_RE = re.compile(r"<think>.*?</think>", re.DOTALL)
_PASSIVE = {"check", "call"}
_AGGRESSIVE = {"bet", "raise"}


def _strip_thinking(text):
    return _THINK_RE.sub("", text).strip()


def parse_action_type(text):
    text = _strip_thinking(text).lower()
    for action in VALID_ACTIONS:
        if text.startswith(action):
            return action
    if "all-in" in text or "allin" in text or "all in" in text:
        return "raise"
    return None


def parse_bet_amount(text):
    text = _strip_thinking(text).lower()
    action = parse_action_type(text)
    if action in ("check", "fold", "call"):
        return None
    match = re.search(r"(\d+(?:\.\d+)?)", text)
    return float(match.group(1)) if match else None


def poker_reward(predicted, correct):
    pred_action = parse_action_type(predicted)
    true_action = parse_action_type(correct)

    if pred_action != true_action:
        both = {pred_action, true_action}
        if both <= _AGGRESSIVE:
            return -0.3
        if both <= _PASSIVE:
            return -0.3
        return -1.0

    true_amount = parse_bet_amount(correct)
    if true_amount is None:
        return 1.0

    if true_amount == 0:
        return 1.0

    pred_amount = parse_bet_amount(predicted)
    if pred_amount is None:
        return 0.1

    ratio = pred_amount / true_amount
    if 0.9 <= ratio <= 1.1:
        return 1.0
    elif 0.8 <= ratio <= 1.2:
        return 0.7
    elif 0.5 <= ratio <= 1.5:
        return 0.4
    else:
        return 0.1


# Sanity check
assert poker_reward("bet 18", "bet 18") == 1.0
assert poker_reward("bet 20", "bet 18") == 0.7  # within 20%
assert poker_reward("fold", "raise 10") == -1.0
assert poker_reward("bet 10", "raise 10") == -0.3  # same category
assert poker_reward("check", "call") == -0.3  # same category
assert poker_reward("fold", "fold") == 1.0
print("Reward function OK — all sanity checks passed")

## 7. Load model

In [ ]:
from unsloth import FastLanguageModel

MAX_SEQ_LENGTH = 1024

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=SFT_ADAPTER_DIR,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
    dtype=None,
)

print(f"Model loaded — VRAM: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

## 8. Apply LoRA adapters

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    bias="none",
    use_gradient_checkpointing=False,
    random_state=42,
)
model.print_trainable_parameters()

## 9. Preprocess dataset for GRPO

In [ ]:
def preprocess_for_grpo(row):
    return {
        "prompt": apply_chat_template(
            format_grpo(row), tokenizer, add_generation_prompt=True
        ),
        "answer": row["output"],
    }


grpo_dataset = train_ds.map(
    preprocess_for_grpo,
    remove_columns=train_ds.column_names,
)
print(f"Preprocessed {len(grpo_dataset):,} rows")
print(f"\nSample prompt (first 200 chars):\n{grpo_dataset[0]['prompt'][:200]}...")
print(f"\nSample answer: {grpo_dataset[0]['answer']}")

## 10. GRPO reward wrapper

In [ ]:
def grpo_reward_fn(completions, answer=None, **kwargs):
    return [poker_reward(c, a) for c, a in zip(completions, answer)]


# Verify wrapper works
test_result = grpo_reward_fn(["bet 18", "fold"], answer=["bet 18", "raise 10"])
assert test_result == [1.0, -1.0], f"Got {test_result}"
print("GRPO reward wrapper OK")

## 11. Completion logger (prints samples every 100 steps)

In [ ]:
from transformers import TrainerCallback


class CompletionLogger(TrainerCallback):
    def on_step_end(self, cb_args, state, control, **kwargs):
        if state.global_step % 100 == 0:
            model_ref = kwargs.get("model")
            if model_ref is None:
                return
            sample = grpo_dataset[0]
            inputs = tokenizer(sample["prompt"], return_tensors="pt").to("cuda")
            with torch.no_grad():
                completions = []
                for _ in range(3):
                    out = model_ref.generate(
                        **inputs,
                        max_new_tokens=16,
                        do_sample=True,
                        temperature=0.7,
                        pad_token_id=tokenizer.eos_token_id,
                    )
                    gen = tokenizer.decode(
                        out[0][inputs["input_ids"].shape[1]:],
                        skip_special_tokens=True,
                    ).strip()
                    reward = poker_reward(gen, sample["answer"])
                    completions.append(f"'{gen}' -> {reward:.1f}")
            print(f"Step {state.global_step} samples: {' | '.join(completions)} (answer: '{sample['answer']}')")


print("Completion logger ready")

## 12. Configure and run GRPO training

**Key hyperparameter changes from previous run:**
- `learning_rate`: 4e-5 (up from 2e-5)
- `beta`: 0.05 (down from 0.1)
- `max_grad_norm`: 0.3 (up from 0.1)
- `num_generations`: 6 (up from 4)
- `gradient_accumulation_steps`: 4 (up from 2)
- `temperature`: 0.7 (forces diverse completions)
- Tiered reward function (7 levels instead of binary)

In [ ]:
from trl import GRPOTrainer, GRPOConfig
import time

# ── Hyperparameters ──
MAX_STEPS = 2000
BATCH_SIZE = 2
GRAD_ACCUM = 4
NUM_GENERATIONS = 6
LEARNING_RATE = 4e-5
BETA = 0.05
MAX_GRAD_NORM = 0.3
CHECKPOINT_DIR = "/kaggle/working/grpo-checkpoints"
OUTPUT_DIR = "/kaggle/working/grpo-adapter"

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

use_bf16 = torch.cuda.is_bf16_supported()
print(f"Precision: {'bf16' if use_bf16 else 'fp16'}")

config = GRPOConfig(
    output_dir=CHECKPOINT_DIR,
    num_train_epochs=1,
    max_steps=MAX_STEPS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    beta=BETA,
    num_generations=NUM_GENERATIONS,
    max_completion_length=32,
    max_prompt_length=512,
    max_grad_norm=MAX_GRAD_NORM,
    temperature=0.7,
    top_p=0.9,
    bf16=use_bf16,
    fp16=not use_bf16,
    gradient_checkpointing=False,
    warmup_steps=50,
    lr_scheduler_type="cosine",
    logging_steps=25,
    save_strategy="steps",
    save_steps=500,
    save_total_limit=2,
    report_to="none",
)

trainer = GRPOTrainer(
    model=model,
    reward_funcs=grpo_reward_fn,
    args=config,
    train_dataset=grpo_dataset,
)
trainer.add_callback(CompletionLogger())

effective_batch = BATCH_SIZE * GRAD_ACCUM * NUM_GENERATIONS
print(f"Effective batch: {BATCH_SIZE}x{GRAD_ACCUM} prompts x {NUM_GENERATIONS} generations = {effective_batch} completions per update")
print(f"\nStarting training for {MAX_STEPS} steps...")

t0 = time.time()
trainer_stats = trainer.train()
elapsed = time.time() - t0

print(f"\nTraining complete — {elapsed / 60:.1f} min")
print(f"Loss: {trainer_stats.metrics['train_loss']:.4f}")

## 13. Save the trained adapter

In [ ]:
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Adapter saved to {OUTPUT_DIR}")
print(f"Files: {os.listdir(OUTPUT_DIR)}")

## 14. Spot check (20 examples)

In [ ]:
FastLanguageModel.for_inference(model)

spot_test = test_ds.select(range(20))
correct = 0

for row in spot_test:
    prompt = apply_chat_template(
        format_grpo(row), tokenizer, add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=32,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    generated = tokenizer.decode(
        output_ids[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True,
    ).strip()
    generated_clean = _THINK_RE.sub("", generated).strip()
    reward = poker_reward(generated_clean, row["output"])
    if reward == 1.0:
        correct += 1
    print(f"Expected: {row['output']:<12} Predicted: {generated_clean:<12} Reward: {reward}")

print(f"\nSpot-check: {correct}/20 ({correct * 5}%)")

## 15. Download adapter

The adapter is saved in `/kaggle/working/grpo-adapter/`. To download it:

1. After the notebook finishes, go to the **Output** tab
2. Click **Save Version** → **Save & Run All**
3. Download the `grpo-adapter` folder from the output

Alternatively, upload it to HuggingFace Hub:

In [ ]:
# Optional: push to HuggingFace Hub
# Uncomment and set your token:

# from huggingface_hub import login
# login(token="hf_YOUR_TOKEN_HERE")
# model.push_to_hub("dominicvdb/poker-grpo-adapter")
# tokenizer.push_to_hub("dominicvdb/poker-grpo-adapter")
# print("Pushed to HuggingFace Hub")